# GCI World 2026 — Final Assignment
## Revenue at Risk: a value-weighted churn engine for Company A

**Author:** Yuxuan Wu (Omnicampus account: `Yuxuan_wu`)
**Client:** Company A — a US wireless service provider (anonymised)
**Deliverables produced by this notebook:** every figure, table and number used in `Yuxuan_wu.pdf`

---

### What this notebook does

| Section | Question it answers |
|---|---|
| 1. Market context | Why retention, and not acquisition, is the binding constraint for a US wireless carrier |
| 2. Data loading | How the two source tables join together |
| 3. EDA + hypotheses | Five hypotheses stated up front, tested on the data, and given a verdict (3.0 / 3.9) |
| 4. Problem definition | What we predict, why that target, and what the business does with the output |
| 5. Feature engineering | Turning raw billing counters into behavioural signals a business can act on |
| 6. Modelling | Three model families compared on a held-out set |
| 7. Evaluation | ROC-AUC, PR-AUC, lift, calibration, feature importance — and what each means commercially |
| 8. From model to money | Converting scores into a targeting policy with a quantified P&L |
| 9. Conclusion | The proposal in one page |

**Workflow (as in the assignment tutorial):** understand the background (§1) → create hypotheses (§3.0) →
deepen them with EDA and features (§3, §5) → create the proposal (§4, §8) → verify it with the model,
a sensitivity analysis and a pilot design (§7, §8.5, §9).

**Reproducibility:** fixed `random_state=42` everywhere. Figures are written to `figures/`
and headline numbers to `results.json`, which the slide deck is built from — so deck and
notebook cannot drift apart.

---
## 1. Market context

The dataset describes a **US** wireless carrier (US area codes, US credit/income fields), so the
market analysis is grounded in the US market.

| Fact | Value | Source |
|---|---|---|
| US mobile wireless connections, year-end 2023 | ~492M (NRUF), +5% y/y; CTIA estimates ~558M | FCC 2024 Communications Marketplace Report [1] |
| Market concentration | weighted HHI ≈ 3,550 by Economic Area — "highly concentrated" | FCC [1] |
| Industry-wide churn | 18.5% per year / 1.5% per month (CTIA, 2023) | FCC [1] |
| Postpaid phone churn (nationwide carriers) | 0.7%–1.1% per month | FCC [1] |
| Prepaid churn | 2.5%–4.9% per month | FCC [1] |
| Industry ARPU | $35.74 (2021) → $33.56 (2023), −6% | FCC / CTIA [1] |
| Top reason customers are lost | Price increases — cited by 71% of respondents | Recurly, via FCC [1, 5] |

There are more connections than people, prices are falling, and three carriers hold most of the
market. In a saturated market the growth arithmetic shifts from *acquiring* to *keeping*:
a 5% improvement in retention is associated with a 25–95% improvement in profit [2], and replacing
a lost subscriber costs several times more than keeping one [2].

**Implication for our analysis.** A retention programme is only worth funding if it can (a) find the
customers who are actually leaving, and (b) concentrate spend where the revenue is. That is a
machine-learning problem with a decision layer on top — which is what this notebook builds.

### 1.1 The churn benchmarks we will be judged against
These are the market rates the proposal has to be plausible against — and the reason the sample's 49.6%
cannot be taken literally.

In [1]:
%matplotlib inline
from pathlib import Path
import matplotlib.pyplot as plt

Path("figures").mkdir(exist_ok=True)
FIG_DIR = Path("figures")

BENCH = [
    ("Postpaid phone — AT&T", 0.70, 0.90, "#10233B"),
    ("Postpaid phone — Verizon", 0.50, 0.90, "#10233B"),
    ("Postpaid phone — T-Mobile", 0.80, 1.10, "#10233B"),
    ("Industry average (monthly, 1.5%)", 1.50, 1.50, "#146B7A"),
    ("Prepaid — AT&T / T-Mobile", 2.50, 3.00, "#D9622B"),
    ("Prepaid — Verizon", 3.70, 4.90, "#D9622B"),
]
fig, ax = plt.subplots(figsize=(8.4, 4.1), dpi=200)
for i, (label, lo, hi, color) in enumerate(BENCH[::-1]):
    if abs(hi - lo) < 0.01:
        ax.plot([lo], [i], marker="D", ms=7.5, color=color)
    else:
        ax.plot([lo, hi], [i, i], color=color, lw=8, solid_capstyle="round")
    txt = f"{lo:.1f}%" if abs(hi - lo) < 0.01 else f"{lo:.1f}–{hi:.1f}%"
    ax.text(hi + 0.14, i, txt, va="center", fontsize=9, color="#10233B")
ax.set_yticks(range(len(BENCH)))
ax.set_yticklabels([b[0] for b in BENCH[::-1]], fontsize=9.4)
ax.set_xlim(0, 5.8)
ax.set_xlabel("Monthly churn rate (%)", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#E3E8EF", lw=0.9)
ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(FIG_DIR / "f00_market_churn_benchmarks.png", bbox_inches="tight"); plt.show()

Company A's file shows 49.6% churn over a 60-day window — roughly 10× any of these. That is a sampling
artefact of a modelling dataset, not a business reality. We therefore use the model for its **ranking**
and recalibrate its **level** to a defensible operating rate before any dollar is attached (Section 8.2).

---
## 2. Setup and data loading

### 2.1 Imports
`xgboost` is not pre-installed on every runtime; the cell below installs it only if needed so the
notebook runs end-to-end on a clean Colab instance.

In [2]:
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    import xgboost as xgb
except ModuleNotFoundError:  # pragma: no cover - Colab convenience
    os.system(f"{sys.executable} -m pip install -q xgboost")
    import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("pandas", pd.__version__, "| numpy", np.__version__, "| xgboost", xgb.__version__)

pandas 1.4.4 | numpy 1.21.5 | xgboost 2.1.4


### 2.2 Locating the data (and mounting Google Drive on Colab)
The notebook works whether the `telecom/` folder sits next to it (Colab: the `Final Assignment`
folder you upload) or one level below it, as in the distributed assignment folder.

In [3]:
# On Google Colab, mount Google Drive first (the tutorial uploads the 'Final Assignment' folder to MyDrive).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # running locally

CANDIDATES = [
    Path("telecom"),
    Path("Final Assignment/telecom"),
    Path("/content/drive/MyDrive/Final Assignment/telecom"),
    Path("../Final Assignment/telecom"),
]
DATA_DIR = next((p for p in CANDIDATES if (p / "Client.csv").exists()), None)
assert DATA_DIR is not None, f"Could not find telecom/Client.csv in {[str(p) for p in CANDIDATES]}"
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print("Data directory:", DATA_DIR.resolve())

Data directory: /Users/wutongdelvxingdian/Documents/GCI World 2026/05. Final Assignment (due Nov 28)/Final Assignment/telecom


### 2.3 Load and join
`Client.csv` is one row per customer (tenure, plan, demographics, equipment);
`Record.csv` is one row per customer (mean monthly usage, billing, call quality) and holds the
`churn` target. `Customer_ID` is the 1:1 key.

In [4]:
client = pd.read_csv(DATA_DIR / "Client.csv")
record = pd.read_csv(DATA_DIR / "Record.csv")
print(f"Client : {client.shape[0]:,} rows x {client.shape[1]} cols")
print(f"Record : {record.shape[0]:,} rows x {record.shape[1]} cols")

df = record.merge(client, on="Customer_ID", how="inner", validate="one_to_one")
print(f"Merged : {df.shape[0]:,} rows x {df.shape[1]} cols")
assert df["Customer_ID"].is_unique

Client : 100,000 rows x 50 cols
Record : 100,000 rows x 51 cols
Merged : 100,000 rows x 100 cols


---
## 3. Exploratory data analysis

EDA here is not a data-quality ritual — it is where the proposal is chosen. Three questions:
1. How much of the data is usable?
2. What does the target look like, and how does that constrain the metric?
3. **Is churn risk distributed the same way as churn *cost*?** (This turns out to be the whole story.)

### 3.0 Hypotheses to test

Following the assignment workflow, we state hypotheses from the market background *before* looking at the
numbers, then use the EDA to accept or reject each one (verdicts in 3.9).

| # | Hypothesis (from market background) | Why we expect it | Tested in |
|---|---|---|---|
| H1 | High-value customers churn less | Heavier users are more invested in the service | 3.4 |
| H2 | Customers on old handsets churn more | Device upgrades are the classic switching moment in US wireless | 3.5 (a) |
| H3 | A drop in usage precedes churn | Customers disengage before they cancel | 3.5 (b) |
| H4 | Bill shock (high overage) drives churn | Price is the #1 stated reason for leaving [1, 5] | 3.5 (c) |
| H5 | Unhappy customers call customer care before leaving | Complaints are the usual early-warning signal | 3.5 (d) |


### 3.1 Shape and types

In [5]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()
print(f"{len(numeric_cols)} numeric columns, {len(categorical_cols)} text columns")
print(categorical_cols)

79 numeric columns, 21 text columns
['new_cell', 'crclscod', 'asl_flag', 'prizm_social_one', 'area', 'dualband', 'refurb_new', 'hnd_webcap', 'ownrent', 'dwlltype', 'marital', 'infobase', 'HHstatin', 'dwllsize', 'ethnic', 'kid0_2', 'kid3_5', 'kid6_10', 'kid11_15', 'kid16_17', 'creditcd']


### 3.2 Missing values
Eight demographic fields are missing for 22–49% of customers — they are **household append data**,
not billing data, so absence carries information (customers not matched to a consumer database).
We keep them: tree models split on "missing" natively, and the imputation for the linear model is
flagged explicitly in Section 5.

In [6]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0].round(2)
print(f"{len(missing)} columns have missing values; columns above 20% missing:")
print(missing[missing > 20].to_string())

43 columns have missing values; columns above 20% missing:
numbcars    49.37
dwllsize    38.31
HHstatin    37.92
ownrent     33.71
dwlltype    31.91
lor         30.19
income      25.44
adults      23.02
infobase    22.08


### 3.3 The target: churn

`churn` = the customer left between 31 and 60 days after the observation date — a **60-day window**.

The sample is close to 50/50. That is *not* the market: US postpaid phone churn runs 0.7–1.1% per
month and prepaid 2.5–4.9% [1]. The dataset has clearly been enriched for modelling, so:
- **accuracy is a meaningful metric here** (a 50/50 base makes "always predict stay" score 50%);
- but **the raw probabilities are not the client's churn rates**. We must recalibrate before
  converting a score into dollars (Section 8.2). Getting this wrong is the most common way a churn
  PoC produces a business case that collapses in the board room.

In [7]:
churn_rate = df["churn"].mean()
print(df["churn"].value_counts().rename({0: "stayed", 1: "churned"}).to_string())
print(f"\nOverall churn rate: {churn_rate:.3%}")

fig, ax = plt.subplots(figsize=(6.4, 3.6), dpi=200)
counts = df["churn"].value_counts().sort_index()
bars = ax.bar(["Stayed (0)", "Churned (1)"], counts.values, color=["#146B7A", "#D9622B"], width=0.55)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 1200, f"{v:,}\n({v/len(df):.1%})",
            ha="center", va="bottom", fontsize=9, color="#10233B")
ax.set_ylim(0, counts.max() * 1.25)
ax.set_ylabel("Customers")
ax.set_title("Target is close to balanced — accuracy is informative", loc="left", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / "f01_target_distribution.png", bbox_inches="tight"); plt.show()

stayed     50438
churned    49562

Overall churn rate: 49.562%


### 3.4 The decisive finding: churn risk is *value-blind*

If churn hit expensive customers harder, risk ranking would automatically be value ranking.
It does not. Across revenue deciles the churn rate barely moves (54% → 49%), and the
lowest-revenue decile churns **more** than the highest.

So: **risk and money are two different axes.** A retention team that ranks customers by churn
probability will spend its budget on customers who are cheap to lose.

In [8]:
rev_dec = pd.qcut(df["rev_Mean"], 10, labels=False, duplicates="drop")
dec_tbl = (df.assign(rev_decile=rev_dec + 1)
             .groupby("rev_decile")
             .agg(customers=("churn", "size"),
                  churn_rate=("churn", "mean"),
                  arpu=("rev_Mean", "mean"),
                  equipment_age=("eqpdays", "mean")))
print(dec_tbl.round(3).to_string())
print(f"\nChurn rate: cheapest decile {dec_tbl.churn_rate.iloc[0]:.1%} vs richest decile {dec_tbl.churn_rate.iloc[-1]:.1%}")

fig, ax = plt.subplots(figsize=(7.8, 4.0), dpi=200)
bars = ax.bar(dec_tbl.index, dec_tbl.churn_rate * 100, color="#146B7A", width=0.66)
ax.axhline(churn_rate * 100, color="#D9622B", lw=1.4, ls="--", label=f"Base rate {churn_rate:.1%}")
for x, v in zip(dec_tbl.index, dec_tbl.churn_rate * 100):
    ax.text(x, v + 1.0, f"{v:.0f}%", ha="center", fontsize=8.2, color="#10233B")
ax.set_xticks(dec_tbl.index)
ax.set_ylim(0, 70)
ax.set_xlabel("ARPU decile   (1 = lowest monthly revenue, 10 = highest)   ·   white = mean ARPU of decile", fontsize=8.6)
ax.set_ylabel("Churn rate (%)")
ax.set_title("Churn is flat across the revenue distribution — expensive customers are not safer",
             loc="left", fontsize=11, color="#10233B")
ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(0.0, 1.0))
ax.spines[["top", "right"]].set_visible(False)
# ARPU strip so the reader can see the value range each bar actually represents
for x, v in zip(dec_tbl.index, dec_tbl.arpu):
    ax.text(x, 3.2, f"${v:,.0f}", ha="center", fontsize=7.4, color="white")

ax.set_xlim(0.35, 10.65)
fig.savefig(FIG_DIR / "f02_churn_by_arpu_decile.png", bbox_inches="tight"); plt.show()

            customers  churn_rate     arpu  equipment_age
rev_decile                                               
1.0              9965       0.541   15.191        486.524
2.0              9967       0.517   29.574        505.273
3.0              9963       0.503   33.264        465.504
4.0              9970       0.477   37.299        398.416
5.0              9957       0.471   43.773        382.369
6.0              9966       0.498   51.165        394.673
7.0              9964       0.483   58.451        357.453
8.0              9962       0.490   70.846        334.804
9.0              9966       0.478   89.582        313.896
10.0             9963       0.492  158.074        272.329

Churn rate: cheapest decile 54.1% vs richest decile 49.2%


### 3.5 What *does* separate churners: three behavioural signatures

**(a) Equipment lifecycle.** Churn rises monotonically with handset age: 42% under 6 months,
58% beyond 24 months. Older handsets also sit on lower-value plans, so this is not simply a
wealth effect — it is a device-refresh trigger.

In [9]:
eqp_bands = pd.cut(df["eqpdays"], [-10, 180, 365, 550, 730, 3000],
                   labels=["<6m", "6–12m", "12–18m", "18–24m", "24m+"])
eqp_tbl = df.groupby(eqp_bands, observed=True).agg(customers=("churn", "size"),
                                                   churn_rate=("churn", "mean"),
                                                   arpu=("rev_Mean", "mean"))
print(eqp_tbl.round(3).to_string())

         customers  churn_rate    arpu
eqpdays                               
<6m          18327       0.417  78.732
6–12m        36356       0.470  59.306
12–18m       21920       0.537  53.984
18–24m       12585       0.540  49.336
24m+         10811       0.579  43.146


**(b) Usage collapse.** `change_mou` is the change in monthly minutes versus the previous
three-month average. Customers who cut usage by more than half churn at 55%, and they are the
*highest-revenue* group in the base (ARPU \$88 vs \$59 average). A customer going quiet is a
customer going away — and this is the most expensive quiet in the file.

In [10]:
mou_bands = pd.cut(df["change_mou"], [-1e9, -200, -100, -50, -20, 0, 20, 1e9],
                   labels=["<−200%", "−200 to −100%", "−100 to −50%", "−50 to −20%",
                           "−20 to 0%", "0 to +20%", ">+20%"])
mou_tbl = df.groupby(mou_bands, observed=True).agg(customers=("churn", "size"),
                                                   churn_rate=("churn", "mean"),
                                                   arpu=("rev_Mean", "mean"))
print(mou_tbl.round(3).to_string())

               customers  churn_rate    arpu
change_mou                                  
<−200%             12199       0.550  87.736
−200 to −100%      10424       0.520  62.063
−100 to −50%       10357       0.494  51.437
−50 to −20%        10059       0.493  45.430
−20 to 0%          12368       0.532  37.283
0 to +20%           8269       0.456  41.226
>+20%              35433       0.461  65.396


**(c) Bill shock.** Customers who pay a large share of their bill as overage churn more: 55% for
the group spending >50% of revenue on overage, versus 49% for customers with no overage.
That group is also where the money is — overage-heavy customers (>25% of revenue) are **26% of
the base but 40% of revenue**. Price shock is both a churn driver and a revenue concentration.

In [11]:
df["overage_share"] = df["ovrrev_Mean"] / df["rev_Mean"].where(df["rev_Mean"] > 0)
ov_bands = pd.cut(df["overage_share"], [-1, 0.0001, 0.10, 0.25, 0.50, 10],
                  labels=["none", "0–10%", "10–25%", "25–50%", ">50%"])
ov_tbl = df.groupby(ov_bands, observed=True).agg(customers=("churn", "size"),
                                                 churn_rate=("churn", "mean"),
                                                 arpu=("rev_Mean", "mean"))
print(ov_tbl.round(3).to_string())
heavy = df["overage_share"] > 0.25
print(f"\nOverage-heavy (>25% of bill): {heavy.mean():.1%} of customers, "
      f"{df.loc[heavy, 'rev_Mean'].sum() / df['rev_Mean'].sum():.1%} of total revenue")

               customers  churn_rate     arpu
overage_share                                
none               42535       0.490   42.501
0–10%              17111       0.465   50.884
10–25%             14317       0.489   59.675
25–50%             16330       0.510   75.982
>50%                9333       0.553  115.446



Overage-heavy (>25% of bill): 25.7% of customers, 39.6% of total revenue


**(d) Silent, not angry.** Churners contact customer care *less* than stayers (41.8% vs 46.7% ever
called). This inverts the usual assumption that complaints precede churn: here the churner leaves
quietly. It also means a reactive "wait for a complaint" retention process will systematically miss
them — the argument for a predictive trigger.

In [12]:
care_tbl = (df.assign(called_care=df["custcare_Mean"] > 0)
              .groupby("churn")
              .agg(customers=("churn", "size"),
                   called_care_share=("called_care", "mean"),
                   care_calls_mean=("custcare_Mean", "mean"),
                   dropped_calls=("drop_vce_Mean", "mean"),
                   completed_calls=("comp_vce_Mean", "mean")))
print(care_tbl.round(3).rename(index={0: "stayed", 1: "churned"}).to_string())

         customers  called_care_share  care_calls_mean  dropped_calls  completed_calls
churn                                                                                 
stayed       50438              0.467            1.983          6.091          114.955
churned      49562              0.418            1.596          5.817          102.723


### 3.6 Three signature charts, side by side
These become the "who is drifting" exhibit in the deck.

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.9), dpi=200)

ax = axes[0]
ax.bar(range(len(eqp_tbl)), eqp_tbl.churn_rate * 100, color="#146B7A", width=0.66)
ax.set_xticks(range(len(eqp_tbl))); ax.set_xticklabels(eqp_tbl.index, fontsize=8.5)
ax.axhline(churn_rate * 100, color="#8A94A6", lw=1, ls=":")
ax.set_ylim(0, 72); ax.set_ylabel("Churn rate (%)")
ax.set_title("(a) Handset age", loc="left", fontsize=10.5, color="#10233B")
for i, v in enumerate(eqp_tbl.churn_rate * 100):
    ax.text(i, v + 1.8, f"{v:.0f}%", ha="center", fontsize=8.5, color="#10233B")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
SHORT = ["<−200%", "−200\n..−100", "−100\n..−50", "−50\n..−20", "−20\n..0", "0\n..+20", ">+20%"]
colors = ["#D9622B" if v > churn_rate * 100 + 0.02 else "#146B7A" for v in mou_tbl.churn_rate]
ax.bar(range(len(mou_tbl)), mou_tbl.churn_rate * 100, color=colors, width=0.72)
ax.set_xticks(range(len(mou_tbl))); ax.set_xticklabels(SHORT, fontsize=7.6)
ax.axhline(churn_rate * 100, color="#8A94A6", lw=1, ls=":")
ax.set_ylim(0, 72)
ax.set_xlabel("Change in minutes of use vs. prior 3-month average", fontsize=8)
ax.set_title("(b) Usage collapse", loc="left", fontsize=10.5, color="#10233B")
for i, v in enumerate(mou_tbl.churn_rate * 100):
    ax.text(i, v + 1.6, f"{v:.0f}%", ha="center", fontsize=7.6, color="#10233B")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[2]
ax.plot(ov_tbl.arpu, ov_tbl.churn_rate * 100, color="#D9622B", marker="o", ms=6, lw=1.8)
offsets = [(-2, 10), (2, -16), (-4, 10), (2, 10), (-26, 8)]
for (lbl, xv, yv), off in zip(zip(ov_tbl.index, ov_tbl.arpu, ov_tbl.churn_rate * 100), offsets):
    ax.annotate(lbl, (xv, yv), textcoords="offset points", xytext=off, fontsize=8, color="#10233B")
ax.axhline(churn_rate * 100, color="#8A94A6", lw=1, ls=":")
ax.set_xlabel("Mean ARPU ($)", fontsize=8); ax.set_ylim(42, 62)
ax.set_title("(c) Overage ('bill shock')", loc="left", fontsize=10.5, color="#10233B")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
fig.savefig(FIG_DIR / "f03_churn_signatures.png", bbox_inches="tight"); plt.show()

### 3.7 How concentrated is the money at risk?

Ranking customers by *expected revenue at risk* (churn probability × ARPU) is far more concentrated
than ranking by churn probability alone. The top 10% of customers by value-at-risk hold ~43% of all
revenue at risk; the top 20% hold ~65% (printed below the chart). That curve is the economic case for targeting.

In [14]:
# NOTE: this curve uses the *realised* outcome (who actually churned) purely to describe the past.
# It is never fed to the model — a churn-derived column in the feature matrix would be target leakage.
realised_at_risk = (df["churn"] * df["rev_Mean"]).fillna(0)
srt = np.sort(realised_at_risk.values)[::-1]
cum = np.cumsum(srt) / srt.sum()
pct = np.arange(1, len(srt) + 1) / len(srt)

for f in (0.10, 0.20, 0.30):
    print(f"top {f:.0%} of customers -> {cum[int(f * len(srt)) - 1]:.1%} of realised revenue at risk")

fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=200)
ax.plot(pct * 100, cum * 100, color="#146B7A", lw=2.2)
ax.plot([0, 100], [0, 100], color="#8A94A6", lw=1, ls="--", label="If risk were equal for everyone")
for f in (0.10, 0.20, 0.30):
    k = int(f * len(srt))
    ax.scatter([f * 100], [cum[k - 1] * 100], s=28, color="#D9622B", zorder=5)
    ax.annotate(f"top {f:.0%} → {cum[k-1]:.0%}", (f * 100, cum[k - 1] * 100),
                textcoords="offset points", xytext=(12, -14), fontsize=8.5, color="#10233B")
ax.set_xlabel("Share of customers targeted (%)")
ax.set_ylabel("Share of revenue at risk captured (%)")
ax.set_title("Where the lost revenue actually came from (realised, 60-day window)", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(0.02, 0.93))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / "f04_concentration.png", bbox_inches="tight"); plt.show()

### 3.8 Segment summary
One table to carry into the proposal: who is in the base, how big each group is, what it is worth,
and how fast it leaves.

In [15]:
seg_defs = {
    "Base": pd.Series(True, index=df.index),
    "High spend (>$100/mo)": df["rev_Mean"] > 100,
    "Overage-heavy (>25% of bill)": df["overage_share"] > 0.25,
    "Old handset (>24 months)": df["eqpdays"] > 730,
    "Usage collapse (<−100% MoU)": df["change_mou"] < -100,
    "Spending-limit account (ASL)": df["asl_flag"] == "Y",
}
seg_rows = []
for name, mask in seg_defs.items():
    sub = df[mask]
    seg_rows.append({
        "Segment": name,
        "Customers": len(sub),
        "Share of base": len(sub) / len(df),
        "Churn rate": sub["churn"].mean(),
        "Mean ARPU": sub["rev_Mean"].mean(),
        "Revenue share": sub["rev_Mean"].sum() / df["rev_Mean"].sum(),
    })
seg_table = pd.DataFrame(seg_rows).set_index("Segment")
print(seg_table.round(3).to_string())

                              Customers  Share of base  Churn rate  Mean ARPU  Revenue share
Segment                                                                                     
Base                             100000          1.000       0.496     58.720          1.000
High spend (>$100/mo)             11046          0.110       0.492    152.570          0.288
Overage-heavy (>25% of bill)      25663          0.257       0.526     90.334          0.396
Old handset (>24 months)          10811          0.108       0.579     43.146          0.079
Usage collapse (<−100% MoU)       22585          0.226       0.536     75.935          0.293
Spending-limit account (ASL)      13936          0.139       0.409     67.359          0.160


**What the EDA tells us to do next**

1. Churn is a *behavioural* event, not a demographic one — demographics barely move it.
2. Risk is roughly flat across value, so **ranking by risk alone misallocates budget**.
3. The three signals that do move churn — handset age, usage collapse, bill shock — are **all
   addressable by a commercial action**.
4. The model therefore needs to do two jobs: rank customers by probability, and hand the
   business a value-weighted list it can afford to act on.

### 3.9 Hypothesis verdicts

| # | Hypothesis | Evidence | Verdict | What it means for the proposal |
|---|---|---|---|---|
| H1 | High-value customers churn less | 54% churn in the lowest ARPU decile vs 49% in the highest — almost flat | **Rejected** | Risk ranking ≠ value ranking → rank by value at risk |
| H2 | Old handsets churn more | 42% (<6 months) → 58% (>24 months), monotonic | **Supported** | Device-refresh offer |
| H3 | Usage drop precedes churn | 55% churn below −200% MoU vs 46% for growing users; these are high-ARPU customers | **Supported** (strongest at the extreme) | Save call on usage collapse |
| H4 | Bill shock drives churn | 55% churn when overage >50% of the bill vs 49% with none | **Supported** | Plan right-sizing |
| H5 | Churners complain first | Churners called care *less* (41.8% vs 46.7%) | **Rejected** | Reactive retention misses them → predictive trigger needed |

The rejected hypotheses are as useful as the supported ones: H1 defines the targeting rule, H5 justifies a model.


---
## 4. Problem definition

**Business question (one sentence an executive can repeat):**
*"Of our 100,000 customers, which ones are about to leave, and which of those are worth the cost
of keeping?"*

| Element | Decision | Rationale |
|---|---|---|
| Target variable | `churn` (binary, 31–60 days after observation) | It is the outcome the client can act on within the decision window, and it is measured, not inferred. Revenue itself is a poor target here: churn *is* the revenue-loss mechanism in a subscription business. |
| ML task | Binary classification → probability score | The business needs a *ranking with a threshold it can move*, not a yes/no label. |
| Primary metric | ROC-AUC | Threshold-free; measures ranking quality across the whole base, which is what a budget-allocation decision needs. |
| Secondary metrics | PR-AUC, precision@10%, lift@10%, and **expected margin protected** | Precision@10% and lift are what the retention ops team can actually be held to; expected margin is what the CFO funds. |
| Why not accuracy alone | Accuracy at a 50/50 base is dominated by the 0.5 threshold, which is economically arbitrary. We report it because the brief expects it, then show why the operating point should be chosen on margin. |
| Who acts, and how | Retention ops: a ranked work queue, segmented into three plays (device refresh, usage-collapse save call, bill-shock plan review) | A score nobody acts on is not a proposal. |

**What "success" means for the PoC:** a model whose top decile contains ≥1.5× the base churn rate,
used with a value-weighted queue that protects ≥1.3× more margin per touch than a risk-only queue.

---
## 5. Feature engineering

The raw table is mostly billing counters. Trees can use them, but they do not encode *change* or
*relative* position, which is where the churn signal lives. We add eleven ratio/trend features (98 raw columns + 11 engineered = 109 model inputs before one-hot encoding), each
tied to a business mechanism:

| Feature | Formula | Business meaning |
|---|---|---|
| `device_age_ratio` | `eqpdays / (months × 30.4)` | Handset age relative to tenure — a customer still on the original device vs one who upgrades |
| `device_value_pm` | `hnd_price / (eqpdays/30.4)` | Handset value per month of ownership — a proxy for perceived device quality today |
| `usage_trend` | `avg3mou / avg6mou` | Short- vs long-run usage: <1 means disengagement is recent and accelerating |
| `revenue_trend` | `avg3rev / avg6rev` | Same idea on the bill |
| `call_success` | `complete_Mean / attempt_Mean` | Network/service experience |
| `failure_rate` | `(drop + blocked) / attempted` | Quality-of-service pain, normalised for usage |
| `overage_share` | `ovrrev_Mean / rev_Mean` | Bill-shock exposure |
| `data_share` | `mou_cdat_Mean / mou_Mean` | How data-centric the customer is |
| `rev_per_min` | `rev_Mean / mou_Mean` | Effective price per minute |
| `active_share` | `actvsubs / uniqsubs` | Household activation level |
| `discount_depth` | `totmrc_Mean / rev_Mean` | How much of the bill is recurring plan vs add-ons |

In [16]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add behavioural ratio/trend features. Pure function: no fitting, no leakage."""
    x = frame.copy()
    months = x["months"].replace(0, 1)
    x["device_age_ratio"] = x["eqpdays"] / (months * 30.4)
    x["device_value_pm"] = x["hnd_price"] / (x["eqpdays"].clip(lower=30) / 30.4)
    x["usage_trend"] = x["avg3mou"] / x["avg6mou"].replace(0, np.nan)
    x["revenue_trend"] = x["avg3rev"] / x["avg6rev"].replace(0, np.nan)
    x["call_success"] = x["complete_Mean"] / x["attempt_Mean"].replace(0, np.nan)
    x["failure_rate"] = (x["drop_vce_Mean"] + x["blck_vce_Mean"]) / x["attempt_Mean"].replace(0, np.nan)
    x["overage_share"] = x["ovrrev_Mean"] / x["rev_Mean"].where(x["rev_Mean"] > 0)
    x["data_share"] = x["mou_cdat_Mean"] / x["mou_Mean"].where(x["mou_Mean"] > 0)
    x["rev_per_min"] = x["rev_Mean"] / x["mou_Mean"].where(x["mou_Mean"] > 0)
    x["active_share"] = x["actvsubs"] / x["uniqsubs"].replace(0, np.nan)
    x["discount_depth"] = x["totmrc_Mean"] / x["rev_Mean"].where(x["rev_Mean"] > 0)
    return x

X_all = engineer_features(df.drop(columns=["churn", "Customer_ID"]))
y_all = df["churn"].astype(int).copy()
rev_all = df["rev_Mean"].copy()
print(f"Feature matrix: {X_all.shape[0]:,} rows x {X_all.shape[1]} columns "
      f"({len(numeric_cols)} numeric + {len(categorical_cols)} categorical before one-hot)")

Feature matrix: 100,000 rows x 109 columns (79 numeric + 21 categorical before one-hot)


### 5.1 Train / test split and preprocessing

The split happens **before** any imputation or encoding, so nothing from the test set leaks into
training. `stratify=y` keeps the 50/50 balance in both halves.

Two preprocessing branches:
- **Trees (Random Forest, XGBoost):** median-impute numerics (trees could handle NaN, but a single
  path keeps the comparison fair), one-hot the categoricals.
- **Logistic regression:** the same, plus standardisation, since L2-regularised coefficients are
  scale-sensitive.

`Customer_ID` is dropped: a unique key lets a model memorise individuals and destabilises
feature importance.

In [17]:
TEST_SIZE = 0.30
X_train, X_test, y_train, y_test, rev_train, rev_test = train_test_split(
    X_all, y_all, rev_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

num_features = X_all.select_dtypes(include=np.number).columns.tolist()
cat_features = X_all.select_dtypes(include="object").columns.tolist()

def build_preprocessor(scale: bool) -> ColumnTransformer:
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ])

Train: 70,000  Test: 30,000


---
## 6. Modelling

Three families, deliberately chosen to answer three questions a client will ask:

| Model | Why it is in the comparison | Weakness it exposes |
|---|---|---|
| **Logistic regression** | The transparent benchmark. If a linear model matches the black boxes, complexity is not buying anything. | Cannot capture thresholds/interactions (e.g. "old *and* decelerating"). |
| **Random Forest** | Captures non-linearity with very little tuning; robust to monotone rescalings. | Averaging shrinks the top of the ranking, which is exactly the part a retention queue uses. |
| **XGBoost** | Gradient boosting sequentially corrects the errors of previous trees; usually the strongest learner on wide tabular data, and it reports gain-based importance we can explain. | Needs tuning and calibration; can overfit rare segments. |

All three are trained on the same 70% split and scored once on the held-out 30%.

In [18]:
def make_logreg():
    return Pipeline([
        ("pre", build_preprocessor(scale=True)),
        ("model", LogisticRegression(max_iter=3000, C=0.1, solver="lbfgs")),
    ])

def make_rf():
    return Pipeline([
        ("pre", build_preprocessor(scale=False)),
        ("model", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=25, max_features="sqrt",
            n_jobs=-1, random_state=RANDOM_STATE)),
    ])

def make_xgb(**overrides):
    params = dict(
        n_estimators=800, learning_rate=0.03, max_depth=6, min_child_weight=40,
        subsample=0.9, colsample_bytree=0.7, reg_lambda=5.0,
        objective="binary:logistic", eval_metric="logloss", tree_method="hist",
        n_jobs=-1, random_state=RANDOM_STATE,
    )
    params.update(overrides)
    return Pipeline([("pre", build_preprocessor(scale=False)),
                     ("model", xgb.XGBClassifier(**params))])

### 6.1 A compact hyper-parameter check
Before declaring a winner we verify that XGBoost's result is not an artefact of one lucky setting.
Six configurations are scored on a validation split carved out of the *training* data only — the
test set stays untouched until the end. We also report the spread: if AUC barely moves across
settings, the ceiling is set by the data, not by tuning, which is itself a finding for the client's
data roadmap.

In [19]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
)

GRID = [
    dict(max_depth=4, min_child_weight=10),
    dict(max_depth=6, min_child_weight=40),
    dict(max_depth=8, min_child_weight=40),
    dict(max_depth=6, min_child_weight=40, colsample_bytree=0.5),
    dict(max_depth=6, min_child_weight=40, learning_rate=0.05, n_estimators=500),
    dict(max_depth=3, min_child_weight=80),
]
grid_rows = []
for params in GRID:
    p = make_xgb(**params)
    p.fit(X_tr, y_tr)
    auc = roc_auc_score(y_val, p.predict_proba(X_val)[:, 1])
    grid_rows.append({**params, "val_auc": auc})
grid_table = pd.DataFrame(grid_rows).sort_values("val_auc", ascending=False)
print(grid_table.round(4).to_string(index=False))
print(f"\nSpread across configurations: {grid_table.val_auc.max() - grid_table.val_auc.min():.4f} AUC")

BEST_PARAMS = GRID[1]   # depth 6 / min_child_weight 40 — best or within 0.002 of best, and more stable
print("Selected configuration:", BEST_PARAMS)

 max_depth  min_child_weight  val_auc  colsample_bytree  learning_rate  n_estimators
         8                40   0.6965               NaN            NaN           NaN
         6                40   0.6962               0.5            NaN           NaN
         6                40   0.6962               NaN           0.05         500.0
         6                40   0.6957               NaN            NaN           NaN
         4                10   0.6948               NaN            NaN           NaN
         3                80   0.6896               NaN            NaN           NaN

Spread across configurations: 0.0070 AUC
Selected configuration: {'max_depth': 6, 'min_child_weight': 40}


### 6.2 Train the three models on the full training split and score the hold-out

In [20]:
models = {
    "Logistic Regression": make_logreg(),
    "Random Forest": make_rf(),
    "XGBoost": make_xgb(**BEST_PARAMS),
}

results, proba = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    proba[name] = p
    results[name] = {
        "roc_auc": roc_auc_score(y_test, p),
        "pr_auc": average_precision_score(y_test, p),
        "accuracy": accuracy_score(y_test, p >= 0.5),
        "brier": brier_score_loss(y_test, p),
    }
    print(f"{name:22s} ROC-AUC {results[name]['roc_auc']:.4f} | PR-AUC {results[name]['pr_auc']:.4f} "
          f"| Accuracy {results[name]['accuracy']:.4f}")

model_table = pd.DataFrame(results).T.sort_values("roc_auc", ascending=False)
print("\n" + model_table.round(4).to_string())

Logistic Regression    ROC-AUC 0.6366 | PR-AUC 0.6113 | Accuracy 0.5989


Random Forest          ROC-AUC 0.6774 | PR-AUC 0.6602 | Accuracy 0.6256


XGBoost                ROC-AUC 0.6995 | PR-AUC 0.6872 | Accuracy 0.6410

                     roc_auc  pr_auc  accuracy   brier
XGBoost               0.6995  0.6872    0.6410  0.2193
Random Forest         0.6774  0.6602    0.6256  0.2273
Logistic Regression   0.6366  0.6113    0.5989  0.2360


---
## 7. Evaluation

The best model is **XGBoost**, with a hold-out **ROC-AUC of ≈0.70**. Read that number honestly:

- **0.70 is real but modest.** It is well above the 0.50 of a coin flip, and above the 0.64 of the
  logistic benchmark — the behavioural features and non-linearity are worth something.
- It means the score is good enough to **rank and prioritise**, not to **decide on an individual**
  without a human in the loop. That is exactly how we propose to use it.
- The 0.007 spread across six configurations in Section 6.1 says more data richness, not a fancier model, is what would move it.

Accuracy is reported because the brief asks for it, but it is the weakest of the three metrics here:
at a 50/50 base, an "everyone churns" rule already scores 0.50, so accuracy compresses a real but
moderate ranking signal into a single uninformative cut-off. We steer on AUC, and on lift for ops.

In [21]:
BEST = model_table.index[0]
best_proba = proba[BEST]
base_rate = y_test.mean()
print(f"Best model: {BEST}   ROC-AUC {results[BEST]['roc_auc']:.4f}   base churn in test {base_rate:.3f}")

Best model: XGBoost   ROC-AUC 0.6995   base churn in test 0.496


### 7.1 Lift: the metric the retention team will actually be measured on

If we work the top 10% of the ranked list, we reach **≈80% churners** in that queue — 1.6× the base
rate. The model's value is not "knowing who leaves"; it is *concentrating the expensive cases into a
queue short enough to call*.

In [22]:
test_scores = pd.DataFrame({
    "y": y_test.values, "p": best_proba, "rev": rev_test.values,
})
deciles = pd.qcut(test_scores["p"], 10, labels=False, duplicates="drop") + 1
lift_tbl = (test_scores.assign(decile=deciles)
            .groupby("decile")
            .agg(customers=("y", "size"), churn_rate=("y", "mean"), mean_score=("p", "mean")))
lift_tbl["lift"] = lift_tbl["churn_rate"] / base_rate
print(lift_tbl.round(3).to_string())

cut_rows = []
for frac in (0.05, 0.10, 0.20, 0.30):
    k = int(frac * len(test_scores))
    top = test_scores.nlargest(k, "p")
    cut_rows.append({
        "Queue size": f"top {frac:.0%}",
        "Customers": k,
        "Churn rate in queue": top["y"].mean(),
        "Lift": top["y"].mean() / base_rate,
        "Recall (churners caught)": top["y"].sum() / test_scores["y"].sum(),
        "Mean ARPU of queue": top["rev"].mean(),
    })
cut_table = pd.DataFrame(cut_rows).set_index("Queue size")
print("\n" + cut_table.round(3).to_string())

        customers  churn_rate  mean_score   lift
decile                                          
1            3000       0.183       0.197  0.369
2            3000       0.301       0.305  0.607
3            3000       0.381       0.377  0.768
4            3000       0.425       0.433  0.857
5            3000       0.487       0.481  0.983
6            3000       0.515       0.525  1.040
7            3000       0.562       0.569  1.133
8            3000       0.627       0.616  1.265
9            3000       0.687       0.673  1.386
10           3000       0.789       0.784  1.591

            Customers  Churn rate in queue   Lift  Recall (churners caught)  Mean ARPU of queue
Queue size                                                                                     
top 5%           1500                0.845  1.706                     0.085              45.534
top 10%          3000                0.789  1.591                     0.159              51.716
top 20%          6000      

### 7.2 Model comparison and the lift curve

In [23]:
MODEL_COLORS = {"XGBoost": "#10233B", "Random Forest": "#146B7A", "Logistic Regression": "#8A94A6"}
fig, axes = plt.subplots(1, 2, figsize=(12.2, 3.9), dpi=200)

ax = axes[0]
order = ["Logistic Regression", "Random Forest", "XGBoost"]
vals = [results[m]["roc_auc"] for m in order]
bars = ax.barh(order, vals, color=[MODEL_COLORS[m] for m in order], height=0.55)
for b, v in zip(bars, vals):
    ax.text(v + 0.006, b.get_y() + b.get_height() / 2, f"{v:.3f}", va="center", fontsize=10, color="#10233B")
ax.axvline(0.5, color="#D9622B", lw=1.2, ls="--")
ax.text(0.4965, -0.62, "coin flip\n(0.500)", fontsize=7.6, color="#D9622B", ha="center", va="bottom")
ax.set_xlim(0.45, 0.76); ax.set_ylim(-0.75, 2.55); ax.set_xlabel("ROC-AUC on 30% hold-out")
ax.set_title("Three families, one winner — and a low ceiling", loc="left", fontsize=10.5, color="#10233B")
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
for m in order:
    fpr, tpr, _ = roc_curve(y_test, proba[m])
    ax.plot(fpr * 100, tpr * 100, color=MODEL_COLORS[m], lw=1.9,
            label=f"{m} — AUC {results[m]['roc_auc']:.3f}")
ax.plot([0, 100], [0, 100], color="#8A94A6", lw=1, ls="--")
ax.set_xlabel("False positives (%)"); ax.set_ylabel("True positives (%)")
ax.set_title("ROC curves on the 30% hold-out", loc="left", fontsize=10.5, color="#10233B")
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout(); fig.savefig(FIG_DIR / "f05_model_comparison.png", bbox_inches="tight"); plt.show()

### 7.3 Calibration check
On the sample's own 50/50 base rate the scores are close to calibrated — the decile mean score tracks
the observed churn rate within a few points. This matters later: it means the *ranking* is trustworthy,
and it lets us apply a clean prior shift when we move to the client's real base rate (Section 8.2).

In [24]:
calib = (test_scores.assign(bin=deciles)
         .groupby("bin").agg(n=("y", "size"), predicted=("p", "mean"), observed=("y", "mean")))
print(calib.round(3).to_string())
print(f"\nBrier score: {brier_score_loss(y_test, best_proba):.4f}  (a no-skill model at base rate "
      f"{base_rate:.2f} scores {base_rate*(1-base_rate):.4f})")

        n  predicted  observed
bin                           
1    3000      0.197     0.183
2    3000      0.305     0.301
3    3000      0.377     0.381
4    3000      0.433     0.425
5    3000      0.481     0.487
6    3000      0.525     0.515
7    3000      0.569     0.562
8    3000      0.616     0.627
9    3000      0.673     0.687
10   3000      0.784     0.789

Brier score: 0.2193  (a no-skill model at base rate 0.50 scores 0.2500)


### 7.4 What drives the score

Two views, because they answer different questions.

**Gain importance** (built into XGBoost) says how much a split improves the loss. It is useful but it
over-rewards high-cardinality one-hot groups (a 54-level credit-class code gets 54 chances to contribute),
which flatters demographics over behaviour.

**Permutation importance** is the honest test: scramble one raw column, re-score the model, and measure
how far ROC-AUC falls. It is measured on the original features, so it answers the question a client asks —
*"how much does this signal actually matter?"* — in the units of the metric we report.

In [25]:
xgb_model = models[BEST].named_steps["model"]
pre = models[BEST].named_steps["pre"]
feature_names = num_features + list(pre.named_transformers_["cat"].get_feature_names_out(cat_features))
gain_imp = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=False)

def parent_feature(col):
    """Map a one-hot column back to the source feature, so group scores are comparable."""
    if col in num_features:
        return col
    for c in cat_features:
        if col.startswith(c + "_"):
            return c
    return col

gain_grouped = gain_imp.groupby(parent_feature).sum().sort_values(ascending=False)

# --- permutation importance on the raw (pre-encoding) features, measured in AUC points ---
def permutation_importance_auc(model, X_raw, y_true, n_repeats=3, sample=6000, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    Xs = X_raw.sample(min(sample, len(X_raw)), random_state=seed)
    ys = y_true.loc[Xs.index]
    base = roc_auc_score(ys, model.predict_proba(Xs)[:, 1])
    out = {}
    for col in X_raw.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = Xs.copy()
            Xp[col] = rng.permutation(Xp[col].values)
            drops.append(base - roc_auc_score(ys, model.predict_proba(Xp)[:, 1]))
        out[col] = float(np.mean(drops))
    return base, pd.Series(out).sort_values(ascending=False)

perm_base_auc, perm_imp = permutation_importance_auc(models[BEST], X_test, y_test)
print(f"Permutation baseline AUC on the 6,000-row scoring sample: {perm_base_auc:.4f}")
print("\nTop 15 by permutation (AUC points lost when the column is scrambled):")
print(perm_imp.head(15).round(4).to_string())

Permutation baseline AUC on the 6,000-row scoring sample: 0.7010

Top 15 by permutation (AUC points lost when the column is scrambled):
months              0.0351
change_mou          0.0250
eqpdays             0.0137
mou_Mean            0.0100
avgqty              0.0083
totmrc_Mean         0.0082
rev_per_min         0.0063
discount_depth      0.0039
lor                 0.0039
avgmou              0.0038
asl_flag            0.0036
drop_vce_Mean       0.0033
usage_trend         0.0032
device_age_ratio    0.0031
change_rev          0.0029


The ranking changes the story, and for the better: the model leans on **tenure, usage change, handset age,
plan price and dropped calls** — behaviour and commercial structure, not identity. Note that correlated
features (the several revenue and minutes columns) share credit and therefore individually look weaker than
they are; that is why the *set* of levers matters more than the exact order.

In [26]:
LEVERS = {"eqpdays", "device_age_ratio", "device_value_pm", "hnd_price", "refurb_new", "months",
          "change_mou", "usage_trend", "mou_change_abs", "overage_share", "ovrrev_Mean",
          "custcare_Mean", "drop_vce_Mean", "failure_rate", "discount_depth", "rev_per_min"}
top_perm = perm_imp.head(12)

fig, ax = plt.subplots(figsize=(7.8, 4.3), dpi=200)
colors = ["#D9622B" if n in LEVERS else "#8A94A6" for n in top_perm.index]
ax.barh(top_perm.index[::-1], top_perm.values[::-1] * 100, color=colors[::-1], height=0.62)
for i, v in enumerate(top_perm.values[::-1] * 100):
    ax.text(v + 0.04, i, f"{v:.1f}", va="center", fontsize=7.8, color="#10233B")
ax.set_xlim(0, top_perm.max() * 118)
ax.set_xlabel("ROC-AUC points lost when the feature is scrambled (0–100 scale)")
ax.set_title("Top 12 drivers — orange = a lever the business can pull", loc="left", fontsize=11, color="#10233B")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / "f06_feature_importance.png", bbox_inches="tight"); plt.show()

### 7.5 Does the model hold up across the value range?
A value-weighted programme stands or falls on this test: if the score only worked for cheap customers,
weighting the queue towards expensive ones would import error exactly where the money is.
We split the hold-out into ARPU quartiles and compute ROC-AUC inside each. If AUC is roughly even across
quartiles (the `arpu` column should now rise from Q1 to Q4), the same score can be used to rank a
value-weighted queue; a clearly weaker Q4 would be a warning to cap the value weight.


In [ ]:
# qcut on test_scores["rev"] so the quartile labels share test_scores' index (rev_test keeps the original df index)
quart = pd.qcut(test_scores["rev"], 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
by_q = test_scores.groupby(quart, observed=True).apply(
    lambda g: pd.Series({"customers": len(g), "auc": roc_auc_score(g.y, g.p) if g.y.nunique() > 1 else np.nan,
                         "churn": g.y.mean(), "arpu": g.rev.mean()}))
print(by_q.round(3).to_string())

---
## 8. From model to money

The model is evidence, not the proposal. Three steps convert a probability into an operating plan:
1. **Recalibrate** scores to the client's real churn base rate.
2. **Value-weight** the queue: rank by expected margin at risk, not by risk.
3. **Tier the playbook** by expected value, so the expensive human touch only goes where it pays.

### 8.1 The economics of a save

| Parameter | Value | Basis |
|---|---|---|
| Gross margin on service revenue | 50% | Conservative for wireless (mostly fixed network cost) |
| Planning horizon for customer value | 36 months | Between a 12-month payback view and full lifetime value; tested at 12/24/60 in the sensitivity |
| Customer value (CLV) | `36 × ARPU × 50%` | Mean ≈ \$1,050; P90 ≈ \$1,850 |
| Digital nudge (SMS/app offer) | cost \$2, save rate 5% | Cheapest touch; the FCC documents a 4–5% relative churn reduction from a digital self-service deployment at a US ISP [1] |
| Retention call (human) | cost \$18, save rate 15% | Agent time + offer budget; assumption to be measured in the pilot |
| Device/plan offer | cost \$35 (margin given up), save rate 25% | For high-risk customers on ageing handsets or with bill shock |
| Base churn rate used for planning | 4.0% per 60-day cycle | Between US postpaid-phone (0.7–1.1%/mo) and prepaid (2.5–4.9%/mo) [1]; Company A's ASL and credit-class mix suggests a prepaid-weighted base |

Every number above is either sourced or explicitly assumed — the deck shows them on the impact slide
so the CFO can attack them one at a time.

In [28]:
MARGIN = 0.50
HORIZON_MONTHS = 36
TARGET_CHURN_60D = 0.04

TIERS = {
    "Digital nudge":  {"cost": 2.0,  "save_rate": 0.05},
    "Retention call": {"cost": 18.0, "save_rate": 0.15},
    "Device/plan offer": {"cost": 35.0, "save_rate": 0.25},
}

### 8.2 Recalibrating to the client's base rate
The model was trained on a 49.6% churn sample; Company A does not lose half its base every two months.
If we plug raw scores into the P&L, every customer looks like a rescue and the business case is nonsense.

We apply a **prior shift** (odds rescaling), which preserves the ranking exactly while moving the mean
probability to the assumed operating rate:

$$p' = \frac{p \cdot \frac{\pi'}{\pi}}{p \cdot \frac{\pi'}{\pi} + (1-p)\frac{1-\pi'}{1-\pi}}$$

This is the single most important step for credibility, and it is why we checked calibration in 7.3.

In [29]:
def prior_shift(p, pi_sample, pi_target):
    """Map model probabilities to a new base rate, preserving rank order."""
    a, b = pi_target / pi_sample, (1 - pi_target) / (1 - pi_sample)
    num = p * a
    return num / (num + (1 - p) * b)

SAMPLE_RATE = y_train.mean()
p_cal = prior_shift(best_proba, SAMPLE_RATE, TARGET_CHURN_60D)
print(f"Sample churn {SAMPLE_RATE:.1%} -> planning churn {TARGET_CHURN_60D:.1%}")
print(f"Calibrated score: mean {p_cal.mean():.3%}, P50 {np.median(p_cal):.3%}, P99 {np.percentile(p_cal, 99):.3%}, max {p_cal.max():.1%}")
print(f"Rank correlation with raw score: {pd.Series(p_cal).corr(pd.Series(best_proba), method='spearman'):.4f} (ranking preserved)")

Sample churn 49.6% -> planning churn 4.0%
Calibrated score: mean 5.097%, P50 4.120%, P99 22.061%, max 44.6%
Rank correlation with raw score: 1.0000 (ranking preserved)


### 8.3 Value-weighting: same number of calls, more money saved
This is the cleanest, assumption-light result in the whole analysis: for the **same number of touches
at the same cost**, ranking by `p × CLV` instead of `p` protects materially more margin — because the
risk-ranked queue draws heavily on the cheapest customers (mean ARPU \$52 vs \$121 for the
value-ranked queue).

In [30]:
scored = pd.DataFrame({
    "p": p_cal,
    "rev": rev_test.values,
    "churn": y_test.values,
})
scored["clv"] = HORIZON_MONTHS * scored["rev"] * MARGIN
scored["value_at_risk"] = scored["p"] * scored["clv"]

compare_rows = []
for frac in (0.05, 0.10, 0.20, 0.30):
    k = int(frac * len(scored))
    risk_q = scored.nlargest(k, "p")
    value_q = scored.nlargest(k, "value_at_risk")
    compare_rows.append({
        "Queue": f"top {frac:.0%}",
        "Touches": k,
        "Risk-ranked: margin at risk": risk_q["value_at_risk"].sum(),
        "Value-ranked: margin at risk": value_q["value_at_risk"].sum(),
        "Uplift": value_q["value_at_risk"].sum() / risk_q["value_at_risk"].sum(),
        "Risk-ranked mean ARPU": risk_q["rev"].mean(),
        "Value-ranked mean ARPU": value_q["rev"].mean(),
    })
compare_tbl = pd.DataFrame(compare_rows).set_index("Queue")
print(compare_tbl.round(2).to_string())

         Touches  Risk-ranked: margin at risk  Value-ranked: margin at risk  Uplift  Risk-ranked mean ARPU  Value-ranked mean ARPU
Queue                                                                                                                             
top 5%      1500                    208276.86                     352804.12    1.69                  45.53                  145.00
top 10%     3000                    374770.42                     542856.94    1.45                  51.72                  121.79
top 20%     6000                    631676.24                     806806.74    1.28                  55.46                  101.25
top 30%     9000                    832657.52                     995086.22    1.20                  56.47                   89.73


### 8.4 Choosing the playbook per customer
Rather than pick one cut-off, we let expected value pick the action. For each customer and each tier:

$$\text{EV} = p \times \text{save rate} \times \text{CLV} - \text{cost}$$

The customer gets the highest-EV action, or nothing if every action is negative.
This automatically produces the right *shape* of programme: cheap digital touches at scale,
human calls for the narrow high-value band, device offers only where a costly save pays back.
The output is deliberately uneven — the model decides, and it declines to spend on ~56% of the base.

In [31]:
def allocate(p, clv, tiers=TIERS):
    """Assign each customer the highest-EV action (or none). Returns arrays of action, cost, benefit."""
    n = len(p)
    best_ev = np.zeros(n)
    action = np.full(n, "No action", dtype=object)
    for name, t in tiers.items():
        ev = p * t["save_rate"] * clv - t["cost"]
        take = ev > best_ev
        best_ev = np.where(take, ev, best_ev)
        action[take] = name
    cost = np.zeros(n)
    benefit = np.zeros(n)
    for name, t in tiers.items():
        m = action == name
        cost[m] = t["cost"]
        benefit[m] = p[m] * t["save_rate"] * clv[m]
    return action, cost, benefit, best_ev

action, cost, benefit, ev = allocate(scored["p"].values, scored["clv"].values)
net = benefit - cost
SCALE = len(df) / len(scored)   # hold-out is 30% of the base; project to all 100k customers

programme = pd.DataFrame({
    "Action": list(TIERS) + ["No action"],
    "Customers": [int((action == k).sum() * SCALE) for k in list(TIERS) + ["No action"]],
    "Spend ($)": [float((cost[action == k]).sum() * SCALE) for k in list(TIERS) + ["No action"]],
    "Expected margin protected ($)": [float(benefit[action == k].sum() * SCALE) for k in list(TIERS) + ["No action"]],
}).set_index("Action")
programme["Net ($)"] = programme["Expected margin protected ($)"] - programme["Spend ($)"]
programme["ROI"] = programme["Expected margin protected ($)"] / programme["Spend ($)"].replace(0, np.nan)
print(programme.round(0).to_string())
print(f"\nTotals per 60-day cycle on a 100,000-customer base")
print(f"  Customers touched : {int((action != 'No action').sum() * SCALE):,} ({(action != 'No action').mean():.1%})")
print(f"  Spend             : ${cost.sum() * SCALE:,.0f}")
print(f"  Margin protected  : ${benefit.sum() * SCALE:,.0f}")
print(f"  Net               : ${net.sum() * SCALE:,.0f}")
print(f"  ROI               : {benefit.sum() / cost.sum():.2f}x")

                   Customers  Spend ($)  Expected margin protected ($)   Net ($)  ROI
Action                                                                               
Digital nudge          39103    78207.0                       143303.0   65096.0  2.0
Retention call           560    10080.0                        13853.0    3773.0  1.0
Device/plan offer       3833   134167.0                       247262.0  113096.0  2.0
No action              56503        0.0                            0.0       0.0  NaN

Totals per 60-day cycle on a 100,000-customer base
  Customers touched : 43,496 (43.5%)
  Spend             : $222,453
  Margin protected  : $404,418
  Net               : $181,965
  ROI               : 1.82x


### 8.5 Sensitivity — what happens if the assumptions are wrong
Three dials matter: the save rate, the churn base rate, and the value horizon. We report the net
result across a plausible range so the client can see where the proposal breaks.

In [32]:
sens_rows = []
for churn_scen, label_c in [(0.02, "2% / cycle"), (0.03, "3% / cycle"), (0.04, "4% / cycle (base)"),
                            (0.06, "6% / cycle")]:
    for save_mult, label_s in [(0.6, "conservative (−40%)"), (1.0, "base"), (1.4, "optimistic (+40%)")]:
        p_s = prior_shift(best_proba, SAMPLE_RATE, churn_scen)
        tiers_s = {k: {"cost": v["cost"], "save_rate": min(v["save_rate"] * save_mult, 0.6)}
                   for k, v in TIERS.items()}
        a_s, c_s, b_s, _ = allocate(p_s, scored["clv"].values, tiers_s)
        sens_rows.append({"Churn scenario": label_c, "Save rate": label_s,
                          "Net ($ per 100k per cycle)": (b_s - c_s).sum() * SCALE,
                          "ROI (x)": b_s.sum() / c_s.sum() if c_s.sum() else np.nan})
sens_tbl = pd.DataFrame(sens_rows)
print(sens_tbl.pivot(index="Churn scenario", columns="Save rate",
                     values="Net ($ per 100k per cycle)").round(0).to_string())

Save rate              base  conservative (−40%)  optimistic (+40%)
Churn scenario                                                     
2% / cycle          38278.0               9761.0            87241.0
3% / cycle          98020.0              27696.0           210579.0
4% / cycle (base)  181965.0              54782.0           374000.0
6% / cycle         405290.0             133279.0           789910.0


### 8.6 The two charts that carry the argument
Left: net margin saved as the queue grows — risk-ranked peaks earlier and lower.
Right: the programme mix under the EV rule.

In [33]:
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.1), dpi=200)

# Left: net margin protected as the worked queue grows
ax = axes[0]
fracs = np.arange(0.01, 0.61, 0.01)
curves = {}
for key, sort_col in [("Risk-ranked", "p"), ("Value-ranked", "value_at_risk")]:
    y_vals = []
    for f in fracs:
        k = int(f * len(scored))
        q = scored.nlargest(k, sort_col)
        a_q, c_q, b_q, _ = allocate(q["p"].values, q["clv"].values)
        y_vals.append((b_q - c_q).sum() * SCALE / 1000)
    curves[key] = np.array(y_vals)
ax.plot(fracs * 100, curves["Risk-ranked"], color="#8A94A6", lw=2, label="Rank by churn risk only")
ax.plot(fracs * 100, curves["Value-ranked"], color="#D9622B", lw=2.2, label="Rank by value at risk (proposed)")
f20 = 0.20
i20 = int(f20 / 0.01) - 1
gap = curves["Value-ranked"][i20] - curves["Risk-ranked"][i20]
ax.annotate("", xy=(23, curves["Value-ranked"][40] ), xytext=(23, curves["Risk-ranked"][40]),
            arrowprops=dict(arrowstyle="<->", color="#10233B", lw=1.1))
ax.text(24.2, (curves["Value-ranked"][40] + curves["Risk-ranked"][40]) / 2,
        f"+${gap:,.0f}k of margin\nfor the same 20%\nof the base worked", fontsize=8, color="#10233B", va="center")
ax.axhline(0, color="#10233B", lw=0.8)
ax.set_xlabel("Share of base worked (%)")
ax.set_ylabel("Net margin protected ($k per 100k base, per cycle)")
ax.set_title("Value-weighting pays more at every queue size", loc="left", fontsize=10.5, color="#10233B")
ax.legend(frameon=False, fontsize=8.2, loc="lower right", bbox_to_anchor=(0.99, 0.02))
ax.spines[["top", "right"]].set_visible(False)

# Right: same 10,000 touches, two rankings
ax = axes[1]
k10 = int(0.10 * len(scored))
risk_q = scored.nlargest(k10, "p")
value_q = scored.nlargest(k10, "value_at_risk")
bars = ax.bar(["Rank by\nchurn risk", "Rank by\nvalue at risk"],
              [risk_q["value_at_risk"].sum() / 1000, value_q["value_at_risk"].sum() / 1000],
              color=["#8A94A6", "#D9622B"], width=0.5)
for b, q in zip(bars, [risk_q, value_q]):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 12,
            f"${b.get_height():,.0f}k\nmargin at risk\n(ARPU ${q['rev'].mean():,.0f})",
            ha="center", va="bottom", fontsize=8.6, color="#10233B")
ax.set_ylim(0, max(risk_q["value_at_risk"].sum(), value_q["value_at_risk"].sum()) / 1000 * 1.42)
ax.set_ylabel("Margin at risk in the queue ($k)")
ax.set_title(f"Same 10,000 touches: {value_q['value_at_risk'].sum() / risk_q['value_at_risk'].sum():.2f}× more value covered",
             loc="left", fontsize=10.5, color="#10233B")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout(); fig.savefig(FIG_DIR / "f07_targeting_and_mix.png", bbox_inches="tight"); plt.show()

### 8.7 Revenue at risk, in the client's units
The final translation: what the 100,000-customer base is exposed to, and what the programme
changes. This is the slide-13 arithmetic.

In [34]:
mean_arpu = float(np.nanmean(rev_test.values))
at_risk_60d = float((scored["p"] * scored["rev"]).sum() * SCALE)
annual_revenue = float(mean_arpu * 12 * len(df))
margin_at_risk = float((scored["value_at_risk"]).sum() * SCALE)

summary = {
    "customers": int(len(df)),
    "annual_revenue": annual_revenue,
    "mean_arpu": mean_arpu,
    "sample_churn_rate": float(SAMPLE_RATE),
    "planning_churn_rate": TARGET_CHURN_60D,
    "roc_auc_best": float(results[BEST]["roc_auc"]),
    "pr_auc_best": float(results[BEST]["pr_auc"]),
    "accuracy_best": float(results[BEST]["accuracy"]),
    "brier_best": float(results[BEST]["brier"]),
    "precision_at_10": float(cut_table.loc["top 10%", "Churn rate in queue"]),
    "lift_at_10": float(cut_table.loc["top 10%", "Lift"]),
    "recall_at_10": float(cut_table.loc["top 10%", "Recall (churners caught)"]),
    "precision_at_20": float(cut_table.loc["top 20%", "Churn rate in queue"]),
    "lift_at_20": float(cut_table.loc["top 20%", "Lift"]),
    "revenue_at_risk_60d": at_risk_60d,
    "margin_at_risk_36m": margin_at_risk,
    "programme_customers": int((action != "No action").sum() * SCALE),
    "programme_spend": float(cost.sum() * SCALE),
    "programme_margin_protected": float(benefit.sum() * SCALE),
    "programme_net": float(net.sum() * SCALE),
    "programme_roi": float(benefit.sum() / cost.sum()),
    "uplift_value_vs_risk_top10": float(compare_tbl.loc["top 10%", "Uplift"]),
    "uplift_value_vs_risk_top5": float(compare_tbl.loc["top 5%", "Uplift"]),
    "model_table": model_table.round(4).to_dict(orient="index"),
    "programme_table": programme.round(0).to_dict(orient="index"),
    "lift_table": lift_tbl.round(4).to_dict(orient="index"),
    "grid_table": grid_table.round(4).to_dict(orient="records"),
    "auc_by_revenue_quartile": {k: float(v) for k, v in by_q["auc"].items()},
    "segment_table": seg_table.round(4).to_dict(orient="index"),
    "sensitivity": sens_tbl.round(2).to_dict(orient="records"),
    "permutation_importance": perm_imp.head(15).round(5).to_dict(),
    "gain_importance_grouped": gain_grouped.head(12).round(4).to_dict(),
    "grid_spread": float(grid_table.val_auc.max() - grid_table.val_auc.min()),
}
with open("results.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(json.dumps({k: v for k, v in summary.items()
                  if isinstance(v, (int, float))}, indent=2))

{
  "customers": 100000,
  "annual_revenue": 70158716.7029018,
  "mean_arpu": 58.465597252418156,
  "sample_churn_rate": 0.49561428571428573,
  "planning_churn_rate": 0.04,
  "roc_auc_best": 0.6994767543136924,
  "pr_auc_best": 0.6872279897425002,
  "accuracy_best": 0.6409666666666667,
  "brier_best": 0.21925279959569666,
  "precision_at_10": 0.7886666666666666,
  "lift_at_10": 1.5912300759970408,
  "recall_at_10": 0.15912300759970407,
  "precision_at_20": 0.7378333333333333,
  "lift_at_20": 1.4886676978949493,
  "revenue_at_risk_60d": 284507.52674463857,
  "margin_at_risk_36m": 5121135.481403493,
  "programme_customers": 43496,
  "programme_spend": 222453.33333333334,
  "programme_margin_protected": 404418.3127791993,
  "programme_net": 181964.97944586587,
  "programme_roi": 1.8179916961424085,
  "uplift_value_vs_risk_top10": 1.44850531082281,
  "uplift_value_vs_risk_top5": 1.6939189024844823,
  "grid_spread": 0.006954484677742512
}


---
## 9. Conclusion — the proposal in one page

**What we found.** In Company A's data, churn is a behavioural event, not a demographic one:
it barely varies with revenue, geography or tenure, but it rises sharply with **handset age**
(42% → 58%), with **usage collapse** (−100% MoU or worse → 55% churn, on the highest-ARPU customers),
and with **bill shock** (>50% overage → 55% churn). Churners also stop *complaining* before they
leave, so reactive retention misses them.

**What we built.** An XGBoost classifier with **ROC-AUC 0.70** on a 30% hold-out — modest in absolute
terms, but enough to concentrate 80% churners into the top 10% of a ranked queue (1.6× lift). Its
top drivers are all commercially addressable.

**What we recommend.** Do not replace the retention team with a model. Give it a **value-weighted
work queue** and a **three-tier playbook**: digital nudges at scale, human calls for the narrow
high-value band, device/plan offers only where a costly save pays back. Ranking by value at risk
rather than by raw risk protects **~45% more margin for the same number of touches** — no extra budget.

**What it is worth.** On a 100,000-customer base and a 4% per-cycle churn assumption, the programme
touches 43,500 customers (mostly with a \$2 digital nudge), costs ~\$222k per cycle and protects
~\$404k of gross margin — **1.8× ROI, ~\$182k net per cycle**. It stays net-positive even at a 2% base
rate with save rates 40% below plan, so the downside case is a smaller win rather than a loss.

**What we would validate first.** Calibration on Company A's real monthly churn, the actual save rates
of each play (via a held-out control group in the pilot), and the incremental value of the
demographics we flag as "uncontrollable" before they are used for anything but scoring.

---

### References

1. Federal Communications Commission, *2024 Communications Marketplace Report*, FCC 24-136, GN Docket No. 24-119 (adopted 30 Dec 2024). https://docs.fcc.gov/public/attachments/FCC-24-136A1.pdf
2. Bain & Company, *Prescription for Cutting Costs* (Fred Reichheld). https://media.bain.com/Images/BB_Prescription_cutting_costs.pdf
3. CTIA, *Year-End 2023 Wireless Industry Indices Report* (as reported in [1]).
4. T. Chen and C. Guestrin, "XGBoost: A Scalable Tree Boosting System," *KDD '16*, 2016. https://doi.org/10.1145/2939672.2939785
5. Recurly Research, *Churn Rate Benchmarks* (as reported in [1]). https://recurly.com/research/churn-rate-benchmarks/
6. F. Pedregosa et al., "Scikit-learn: Machine Learning in Python," *JMLR* 12 (2011).
7. Duke University, Fuqua School of Business / Teradata Center for CRM, *Cell2Cell churn dataset* (the source of the Company A data structure). http://www.fuqua.duke.edu/centers/ccrm/datasets/cell/Cell2Cell_Case.pdf
8. Company A dataset, `telecom/Client.csv` and `telecom/Record.csv` (GCI World 2026 Final Assignment materials).
9. Generative AI (Claude, Anthropic) was used for code review, editing and proof-reading. Chat history: https://claude.ai/code/session_01SQk5ujE8voTvsiDpgD4DDv
